# PatchSight: extraction runs on Kaggle

Serves Qwen3.8-27B (4-bit GGUF, Q4_K_M) with llama.cpp on two T4 GPUs and runs PatchSight's
extractor. **Set `MODE` in section 1** (the notebook stops at once if it is not set):

- **`dunlap`**: the Dunlap Go benchmark (280 reports). Fetches fix commits from GitHub.
- **`pypi`**: the PyPI backfill, one shard per run. Works **offline** from a private input
  bundle, so it makes no GitHub requests.
- **`holdout`**: the Go temporal holdout, 83 reports published after the models' weights
  ([ADR 0010](../docs/adr/0010-go-temporal-holdout.md)). Fetches fix commits from GitHub.
  Run it once with `MODEL = "qwen"`, and once with `MODEL = "gpt-oss-20b"` if ADR 0007 adopts it.
- **`locator`**: the fix-commit locator agent on its masked sample of 200 vulnerabilities
  ([ADR 0011](../docs/adr/0011-fix-commit-locator-agent.md)). Reads GitHub, and needs the
  private input bundle for the pinned OSV snapshot.
- **`topup`**: the prompts the switch-on batch changes (output cap, smallest-first order,
  dotted entry points), listed in `configs/experiments/llm-switch-on-topup.json`: 32 Dunlap
  prompts (reads GitHub) and 794 PyPI prompts (offline, from the private input bundle).
  Stops within the session like `pypi` mode; add its zip as input to resume.

**Before running**

1. *Settings → Accelerator:* **GPU T4 x2**. *Settings → Internet:* **on** (for llama.cpp, the
   model and the tokenizer).
2. *`dunlap`, `holdout`, `locator` and `topup` modes:* *Add-ons → Secrets:* add **`GITHUB_TOKEN`** and attach it to this notebook.
   A GitHub token with no scopes is enough: it only reads public commits.
3. *`pypi`, `locator` and `topup` modes:* *Add Input:* the **private** dataset holding `patchsight-pypi-inputs.zip`
   (made by `export-kaggle-inputs`; it contains third-party code, so never make it public).
   To resume a shard or a locator run that ran out of time, also add that run's output as input. Set `SHARD` (`pypi` mode).
4. Use **Save Version → Save & Run All (Commit)**, so the run continues in the background
   for up to 12 hours.

**Output:** `/kaggle/working/patchsight-kaggle-run.zip` (`dunlap`),
`patchsight-kaggle-run-holdout-<MODEL>.zip` (`holdout`), `patchsight-kaggle-run-locator-<MODEL>.zip`
(`locator`) or `patchsight-kaggle-run-pypi-shard<K>of<N>.zip` (`pypi`), holding the model responses, the
run's summary and `run-info.json`. Import it on your machine with
`python -m patchsight.pipeline import-kaggle-run <zip>`, then re-run the same extraction
command locally with `--role kaggle_extractor`: it replays from the cache with no model calls.

**Expected time:** setup about 20 minutes. `dunlap`: a few hours for all 280 reports.
`topup`: about 7 hours (826 prompts at about 30 seconds each).
`pypi`: the run stops starting new work after `MAX_MINUTES` of extraction, or
`SAFETY_MARGIN_MINUTES` before Kaggle's 12-hour limit (counted from session start), whichever
comes first, so the results are always packaged. A shard that is not finished is resumed by
running it again with this run's output added as input.

## 1. Pinned versions

In [ ]:
import time

NOTEBOOK_STARTED = time.time()
PATCHSIGHT_REPO = "https://github.com/professor3333/PatchSight.git"
PATCHSIGHT_REF = "main"
LLAMA_CPP_TAG = "b12024"  # released 2026-10-31
LLAMA_SOURCE = "prebuilt"  # "prebuilt" (official CUDA 12.8 binaries) or "build" (compile here)
# Published digests of the official release assets.
LLAMA_SHA256 = "2214b85fa76676e395c4928bc5cbcb3536e2a33f83783fffdaca3ce0c11c8acc"
CUDART_SHA256 = "2bf29bb45a323a3633ddb323a906457f03041d93ed30ad8cee67150f2dac54cb"
LLAMA_ARCHIVES = {
    f"llama-{LLAMA_CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz": LLAMA_SHA256,
    f"cudart-llama-{LLAMA_CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz": CUDART_SHA256,
}
# "qwen": the extractor. "gpt-oss-20b": the proposed self-hosted second opinion (ADR 0007),
# dunlap mode only until it is adopted.
MODEL = "qwen"
MODELS = {
    "qwen": (
        "ggml-org/Qwen3.8-27B-GGUF",
        "c1869413dece09109e55b7dcc2c77ef382b0d0bd",
        "Qwen3.8-27B-Q4_K_M.gguf",
        "d15d4b6de55f662ac40e867702d31afe738d032fa06b829b2b1a21a49d4a48ee",
        "kaggle_extractor",
        "extract-v1-qwen3.8-27b-q4km",
    ),
    "gpt-oss-20b": (
        "ggml-org/gpt-oss-20b-GGUF",
        "41e8a94d15d62070b36cf4553a0250717ecf46b8",
        "gpt-oss-20b-MXFP4.gguf",
        "27cd6c432c7672cb812a92f611cf3ba7bbc35928262bb1e1253ff4ee6ae35901",
        "kaggle_second_opinion",
        "extract-v1-gpt-oss-20b-mxfp4",
    ),
}
MODEL_REPO, MODEL_REVISION, MODEL_FILE, MODEL_SHA256, ROLE, TAG = MODELS[MODEL]
MODE = ""  # set this: "pypi" (the PyPI backfill), "dunlap" (the Go benchmark) or
# "holdout" (the Go temporal holdout: 83 reports published after the models, ADR 0010) or
# "locator" (the fix-commit locator agent on its masked sample, ADR 0011) or
# "topup" (the prompts the switch-on batch changes, TOPUP_PLAN, under TOPUP_CONFIG)
TOPUP_CONFIG = "configs/experiments/llm-switch-on.toml"
TOPUP_PLAN = "configs/experiments/llm-switch-on-topup.json"
SPLIT = "all"  # dunlap mode: dev, test or all
# pypi mode: one shard per run, and a time budget that leaves room for setup and packaging
# inside Kaggle's 12-hour limit.
SHARD = ""  # pypi mode: set this, e.g. "2/4"
# Extraction stops starting new work at the earlier of MAX_MINUTES after it begins and
# SAFETY_MARGIN_MINUTES before Kaggle's 12-hour session limit (counted from this cell), so
# packaging always finishes in time, however long the setup took.
# Shard 1 of 4 needed 9.7 hours of extraction (913 calls, about 38 seconds each).
MAX_MINUTES = 630
SESSION_LIMIT_MINUTES = 12 * 60
SAFETY_MARGIN_MINUTES = 45
WORK = "/tmp/patchsight"

# Fail now, before any GPU time is spent, rather than run the wrong job for hours.
assert MODE in ("pypi", "dunlap", "holdout", "locator", "topup"), "set MODE above"
assert MODE != "pypi" or SHARD.count("/") == 1, 'set SHARD above, e.g. "2/4"'
assert MODE not in ("pypi", "topup") or MODEL == "qwen", "this mode runs the extractor (qwen)"
print(
    f"mode {MODE}, model {MODEL}" + (f", shard {SHARD}" if MODE == "pypi" else f", split {SPLIT}")
)

## 2. Check the GPUs and disk

In [ ]:
import os
import shutil
import subprocess

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
)
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")
assert gpus >= 2, "set Accelerator to GPU T4 x2"
os.makedirs(WORK, exist_ok=True)
free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"free disk at {WORK}: {free_gb:.0f} GB")
assert free_gb > 30, "need about 30 GB free for the model and the build"

## 3. Get llama.cpp

The official prebuilt CUDA binaries are used by default, verified against their published
SHA-256. If they cannot see the GPUs, llama.cpp is built from source for the T4 (compute
capability 7.5). Either way, any failure prints the log that explains it.

In [ ]:
import glob
import hashlib
import tarfile
import urllib.request


def sha256(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()


smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(
    next(
        (line for line in smi.splitlines() if "CUDA Version" in line), "nvidia-smi header not found"
    )
)
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
)

LLAMA_SERVER = None
server_env = dict(os.environ)
if LLAMA_SOURCE == "prebuilt":
    base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_CPP_TAG}/"
    unpacked = f"{WORK}/llama-prebuilt"
    for name, expected in LLAMA_ARCHIVES.items():
        archive_path = f"{WORK}/{name}"
        if not os.path.exists(archive_path):
            urllib.request.urlretrieve(base + name, archive_path)
        assert sha256(archive_path) == expected, f"{name} does not match its published SHA-256"
        with tarfile.open(archive_path) as archive:
            archive.extractall(unpacked, filter="data")
    library_dirs = sorted(
        {os.path.dirname(p) for p in glob.glob(f"{unpacked}/**/*.so*", recursive=True)}
    )
    server_env["LD_LIBRARY_PATH"] = ":".join([*library_dirs, os.environ.get("LD_LIBRARY_PATH", "")])
    candidates = glob.glob(f"{unpacked}/**/llama-server", recursive=True)
    if candidates:
        check = subprocess.run(
            [candidates[0], "--list-devices"], env=server_env, capture_output=True, text=True
        )
        print((check.stdout + check.stderr)[-1500:])
        if check.returncode == 0 and "CUDA" in check.stdout + check.stderr:
            LLAMA_SERVER = candidates[0]
    print("prebuilt binaries usable:", LLAMA_SERVER is not None)

In [ ]:
def run_logged(command: list[str], log_path: str, cwd: str) -> None:
    with open(log_path, "w") as log:
        result = subprocess.run(command, cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        with open(log_path) as log:
            print("".join(log.readlines()[-80:]))
        raise RuntimeError(f"{command[0]} failed; the last 80 lines of {log_path} are above")


if LLAMA_SERVER is None:
    for tool in (["nvcc", "--version"], ["cmake", "--version"]):
        try:
            found = subprocess.run(tool, capture_output=True, text=True).stdout.strip()
            print(found.splitlines()[-1] if found else f"{tool[0]}: no output")
        except FileNotFoundError:
            print(f"{tool[0]}: not installed")
    source = f"{WORK}/llama.cpp"
    if not os.path.isdir(source):
        subprocess.run(
            [
                "git",
                "clone",
                "-q",
                "--depth",
                "1",
                "--branch",
                LLAMA_CPP_TAG,
                "https://github.com/ggml-org/llama.cpp.git",
                source,
            ],
            check=True,
        )
    run_logged(
        [
            "cmake",
            "-B",
            "build",
            "-DGGML_CUDA=ON",
            "-DCMAKE_CUDA_ARCHITECTURES=75",
            "-DGGML_NATIVE=OFF",
            "-DLLAMA_CURL=OFF",
            "-DCMAKE_BUILD_TYPE=Release",
        ],
        f"{WORK}/cmake-configure.log",
        source,
    )
    run_logged(
        [
            "cmake",
            "--build",
            "build",
            "--config",
            "Release",
            "-j",
            str(os.cpu_count()),
            "--target",
            "llama-server",
        ],
        f"{WORK}/cmake-build.log",
        source,
    )
    LLAMA_SERVER = f"{source}/build/bin/llama-server"
    server_env = dict(os.environ)
print("llama-server:", LLAMA_SERVER)
print(
    subprocess.run(
        [LLAMA_SERVER, "--version"], env=server_env, capture_output=True, text=True
    ).stderr.strip()[-300:]
)

## 4. Download the model and verify it against the pinned SHA-256

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    MODEL_REPO, MODEL_FILE, revision=MODEL_REVISION, local_dir=f"{WORK}/models"
)
assert sha256(model_path) == MODEL_SHA256, "the model file does not match the pinned SHA-256"
print("model verified:", model_path)

## 5. Start llama-server

Layers are split across both GPUs. The context of 12,288 tokens covers the largest prompt
(about 5,500 tokens), the answer (1,500) and one repair round. Thinking mode is switched off
per request through the chat template (see `configs/llm.toml`).

In [ ]:
import secrets
import time
import urllib.request

os.environ["LLAMA_SERVER_KEY"] = secrets.token_hex(16)
server_log = open(f"{WORK}/llama-server.log", "w")
server = subprocess.Popen(
    [
        LLAMA_SERVER,
        "-m",
        model_path,
        "--host",
        "127.0.0.1",
        "--port",
        "8080",
        "--n-gpu-layers",
        "999",
        "--split-mode",
        "layer",
        "--ctx-size",
        "12288",
        "--parallel",
        "1",
        "--jinja",
        "--seed",
        "20261114",
        "--api-key",
        os.environ["LLAMA_SERVER_KEY"],
    ],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=server_env,
)
for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError("llama-server exited; read llama-server.log")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5) as response:
            if response.status == 200:
                break
    except OSError:
        pass
    time.sleep(5)
else:
    raise RuntimeError("llama-server did not become healthy within 15 minutes")
print("llama-server is up")

## 6. Smoke test: one schema-constrained answer with thinking off

In [ ]:
import json

schema = {
    "type": "object",
    "additionalProperties": False,
    "required": ["answer"],
    "properties": {"answer": {"type": "string"}},
}
body = {
    "model": "local",
    "temperature": 0,
    "seed": 1,
    "max_tokens": 50,
    "messages": [{"role": "user", "content": "Reply with the word ready."}],
    "response_format": {
        "type": "json_schema",
        "json_schema": {"name": "smoke", "schema": schema, "strict": True},
    },
    "chat_template_kwargs": {"enable_thinking": False},
}
request = urllib.request.Request(
    "http://127.0.0.1:8080/v1/chat/completions",
    data=json.dumps(body).encode(),
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.environ['LLAMA_SERVER_KEY']}",
    },
)
try:
    with urllib.request.urlopen(request, timeout=300) as response:
        reply = json.load(response)
except OSError as error:
    with open(f"{WORK}/llama-server.log") as handle:
        print("".join(handle.readlines()[-40:]))
    raise RuntimeError(
        "the smoke test failed; if the log above mentions 'no kernel image', set "
        "LLAMA_SOURCE = 'build' in section 1 and run all cells again"
    ) from error
content = reply["choices"][0]["message"]["content"]
print(content, reply["usage"])
assert "answer" in json.loads(content)

## 7. Install PatchSight and load the GitHub token

In [ ]:
%%bash -s "$PATCHSIGHT_REPO" "$PATCHSIGHT_REF" "$WORK"
set -euo pipefail
cd "$3"
[ -d repo ] || git clone -q "$1" repo
cd repo
git fetch -q origin
git checkout -q "$2"
git log -1 --format='PatchSight %H %s'
pip install -q uv
uv sync -q --all-extras --frozen

In [ ]:
if MODE in ("dunlap", "holdout", "locator", "topup"):
    import urllib.error
    import urllib.request

    from kaggle_secrets import UserSecretsClient

    os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    print("GITHUB_TOKEN loaded")  # never print the value
    # Ask GitHub whether it accepts the token before hours of work depend on it.
    check = urllib.request.Request(
        "https://api.github.com/rate_limit",
        headers={"Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}",
                 "Accept": "application/vnd.github+json"},
    )  # fmt: skip
    try:
        with urllib.request.urlopen(check, timeout=30) as answer:
            left = json.load(answer)["resources"]["core"]["remaining"]
    except urllib.error.HTTPError as refused:
        raise SystemExit(
            f"GitHub refuses GITHUB_TOKEN (HTTP {refused.code}): replace the secret with a valid "
            "token (no scopes needed) and run again"
        ) from None
    print(f"GitHub accepts the token: {left} core requests left this hour")
else:
    print("pypi mode runs offline: no GitHub token needed")

## 8. Run extraction

*`dunlap`:* `fetch-dunlap` checks out the pinned dataset commit and verifies its hashes. The
extraction fetches each fix commit from GitHub (about 1,800 requests) and calls the local
model once per report, plus one repair attempt where needed. The dev and test reports are then
built from the cache, with no further model calls.

*`pypi`:* `restore-kaggle-inputs` puts the bundle's OSV dump and cached responses in place and
verifies the dump's SHA-256. Answers from earlier runs attached as input are imported first, so
a resumed shard skips them. `pypi-extract --offline` then runs the shard.


In [ ]:
REPO = f"{WORK}/repo"


def pipeline(*args: str) -> None:
    command = ["uv", "run", "--frozen", "python", "-m", "patchsight.pipeline", *args]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)


if MODE == "dunlap":
    pipeline("fetch-dunlap")
    pipeline(
        "dunlap-extract",
        "--role",
        ROLE,
        "--split",
        SPLIT,
        "--out",
        f"docs/reports/extraction-dunlap-{SPLIT}-{TAG}",
    )
    # With SPLIT = "all", also write the dev and test reports, from the cache (no model calls).
    # A single split runs only that split: the test split is run once, by a declared decision.
    if SPLIT == "all":
        for part in ("dev", "test"):
            pipeline(
                "dunlap-extract",
                "--role",
                ROLE,
                "--split",
                part,
                "--out",
                f"docs/reports/extraction-dunlap-{part}-{TAG}",
            )
elif MODE == "holdout":
    # The frozen holdout is in the repository; fix commits come from GitHub, like dunlap.
    pipeline(
        "dunlap-extract",
        "--benchmark",
        "go-holdout-v1",
        "--role",
        ROLE,
        "--out",
        f"docs/reports/extraction-go-holdout-v1-{TAG}",
    )
elif MODE == "locator":
    # The private bundle holds the pinned OSV snapshot the masked sample was drawn from.
    manifests = sorted(glob.glob("/kaggle/input/**/patchsight-pypi-inputs.json", recursive=True))
    bundles = sorted(glob.glob("/kaggle/input/**/patchsight-pypi-inputs.zip", recursive=True))
    assert manifests or bundles, "add the private patchsight-pypi-inputs dataset as input"
    pipeline("restore-kaggle-inputs", os.path.dirname(manifests[0]) if manifests else bundles[0])
    # Resuming: an earlier locator run's zip, added as input, brings its cached answers and the
    # GitHub responses it fetched, so finished cases replay at once.
    earlier = glob.glob("/kaggle/input/**/patchsight-kaggle-run-locator*.zip", recursive=True)
    for archive in sorted(earlier):
        pipeline("import-kaggle-run", archive)
    elapsed = (time.time() - NOTEBOOK_STARTED) / 60
    budget = min(MAX_MINUTES, SESSION_LIMIT_MINUTES - SAFETY_MARGIN_MINUTES - elapsed)
    print(f"setup took {elapsed:.0f} min; the agent starts no new case after {budget:.0f} min")
    # Only GitHub answers fetched from here on are packaged: the restored bundle is not.
    FETCHED_AFTER = time.time()
    pipeline(
        "locator-agent-eval",
        "--role",
        ROLE,
        "--tag",
        TAG.replace("extract-v1", "locate-v1"),
        "--max-minutes",
        f"{budget:.0f}",
    )
elif MODE == "pypi":
    # The private input bundle, as Kaggle extracted it or still as the zip.
    manifests = sorted(glob.glob("/kaggle/input/**/patchsight-pypi-inputs.json", recursive=True))
    bundles = sorted(glob.glob("/kaggle/input/**/patchsight-pypi-inputs.zip", recursive=True))
    assert manifests or bundles, "add the private patchsight-pypi-inputs dataset as input"
    pipeline("restore-kaggle-inputs", os.path.dirname(manifests[0]) if manifests else bundles[0])
    previous = glob.glob("/kaggle/input/**/patchsight-kaggle-run*.zip", recursive=True)
    for archive in sorted(previous):
        pipeline("import-kaggle-run", archive)
    elapsed = (time.time() - NOTEBOOK_STARTED) / 60
    budget = min(MAX_MINUTES, SESSION_LIMIT_MINUTES - SAFETY_MARGIN_MINUTES - elapsed)
    print(f"setup took {elapsed:.0f} minutes; extraction budget {budget:.0f} minutes", flush=True)
    assert budget > 30, "too little session time left; start a fresh run"
    pipeline(
        "pypi-extract",
        "--role",
        "kaggle_extractor",
        "--offline",
        "--shard",
        SHARD,
        "--max-minutes",
        f"{budget:.0f}",
        "--out",
        "pypi-run/pypi-extract",
    )
elif MODE == "topup":
    # The prompts the switch-on batch changes (TOPUP_PLAN), under its configuration. Dunlap's
    # read GitHub; the PyPI ones work offline from the private input bundle. Every other
    # prompt keeps its cached answer, replayed on the owner's machine.
    manifests = sorted(glob.glob("/kaggle/input/**/patchsight-pypi-inputs.json", recursive=True))
    bundles = sorted(glob.glob("/kaggle/input/**/patchsight-pypi-inputs.zip", recursive=True))
    assert manifests or bundles, "add the private patchsight-pypi-inputs dataset as input"
    pipeline("restore-kaggle-inputs", os.path.dirname(manifests[0]) if manifests else bundles[0])
    earlier = glob.glob("/kaggle/input/**/patchsight-kaggle-run-topup*.zip", recursive=True)
    for archive in sorted(earlier):
        pipeline("import-kaggle-run", archive)
    pipeline("fetch-dunlap")
    pipeline(
        "dunlap-extract",
        "--role",
        ROLE,
        "--split",
        "all",
        "--config",
        TOPUP_CONFIG,
        "--topup",
        TOPUP_PLAN,
        "--out",
        f"docs/reports/extraction-dunlap-topup-{TAG}",
    )
    elapsed = (time.time() - NOTEBOOK_STARTED) / 60
    budget = min(MAX_MINUTES, SESSION_LIMIT_MINUTES - SAFETY_MARGIN_MINUTES - elapsed)
    print(f"setup took {elapsed:.0f} minutes; extraction budget {budget:.0f} minutes", flush=True)
    assert budget > 30, "too little session time left; start a fresh run"
    pipeline(
        "pypi-extract",
        "--role",
        ROLE,
        "--offline",
        "--config",
        TOPUP_CONFIG,
        "--topup",
        TOPUP_PLAN,
        "--max-minutes",
        f"{budget:.0f}",
        "--out",
        "pypi-run/pypi-extract-topup",
    )
else:
    raise ValueError(f"MODE must be dunlap, holdout, locator, pypi or topup, not {MODE!r}")

## 9. Package the results

In [ ]:
import glob
import zipfile

run_info = {
    "patchsight_commit": subprocess.run(
        ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True
    ).stdout.strip(),
    "llama_cpp_tag": LLAMA_CPP_TAG,
    "model_repo": MODEL_REPO,
    "model_revision": MODEL_REVISION,
    "model_file": MODEL_FILE,
    "model_sha256": MODEL_SHA256,
    "gpus": subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    .stdout.strip()
    .splitlines(),
    "mode": MODE,
    "model": MODEL,
    "role": ROLE,
    "split": SPLIT if MODE == "dunlap" else "go-holdout-v1" if MODE == "holdout" else None,
    "shard": SHARD if MODE == "pypi" else None,
}
shard_name = SHARD.replace("/", "of")
archive = (
    "/kaggle/working/patchsight-kaggle-run.zip"
    if MODE == "dunlap"
    else f"/kaggle/working/patchsight-kaggle-run-holdout-{MODEL}.zip"
    if MODE == "holdout"
    else f"/kaggle/working/patchsight-kaggle-run-locator-{MODEL}.zip"
    if MODE == "locator"
    else f"/kaggle/working/patchsight-kaggle-run-topup-{MODEL}.zip"
    if MODE == "topup"
    else f"/kaggle/working/patchsight-kaggle-run-pypi-shard{shard_name}.zip"
)
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    bundle.writestr("run-info.json", json.dumps(run_info, indent=2))
    with open(f"{WORK}/llama-server.log") as handle:
        bundle.writestr("llama-server.log", "".join(handle.readlines()[-300:]))
    # The run's trace: one line per model call and per extraction, with timings.
    for path in glob.glob(f"{REPO}/data/interim/traces/*.jsonl"):
        bundle.write(path, os.path.relpath(path, REPO))
    for path in glob.glob(f"{REPO}/.cache/llm/**/*", recursive=True):
        if os.path.isfile(path):
            bundle.write(path, os.path.relpath(path, REPO))
    if MODE in ("dunlap", "holdout"):
        reports = "extraction-dunlap-*" if MODE == "dunlap" else "extraction-go-holdout-v1"
        for path in glob.glob(f"{REPO}/docs/reports/{reports}-{TAG}.*"):
            bundle.write(path, os.path.relpath(path, REPO))
    elif MODE == "locator":
        for path in glob.glob(f"{REPO}/docs/reports/locator-masked-v1-*.*"):
            bundle.write(path, os.path.relpath(path, REPO))
        # The GitHub answers this run fetched, so a local replay sees exactly the same.
        for path in glob.glob(f"{REPO}/.cache/http/github/**/*.json", recursive=True):
            if os.path.getmtime(path) >= FETCHED_AFTER:
                bundle.write(path, os.path.relpath(path, REPO))
    else:
        if MODE == "topup":
            for path in glob.glob(f"{REPO}/docs/reports/extraction-dunlap-topup-{TAG}.*"):
                bundle.write(path, os.path.relpath(path, REPO))
        for path in glob.glob(f"{REPO}/pypi-run/*"):
            bundle.write(path, os.path.relpath(path, REPO))
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")
print(json.dumps(run_info, indent=2))

In [ ]:
server.terminate()